# SC22 - Qualitative End-to-End Demonstration

Runs YOLO11 detection, cropping, the shared ConvNeXt-S four-head classifier, item-level view aggregation, and disposal decision rules on the test items. The annotated showcase uses exactly **item_001, item_101, and item_049**, in that order. All successful per-item predictions remain in the CSV; showcase selection is presentation, rather than a summary of overall performance.

**Inputs:** `thesis-common-10c`, raw images and manifests, SC12 output (`weights_convnext_small.fb_in22k_ft_in1k.pt`), and the intended SC8 detector output (`det/weights/best.pt`). Use a suitable GPU.

Calibration is optional in the source. The supplied SC6 calibration belongs to SC7 fusion and must not be applied to SC12. Omit it, or provide calibration separately fitted and identified for the deployed SC12 classifier.


In [ ]:
import sys, glob, os, json
import numpy as np

import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one



_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| threat levels:', T.NUM_THREAT)
OUT = '/kaggle/working'
os.makedirs(OUT, exist_ok=True)

In [ ]:
# ---- load detector (must be the fine-tuned 1-class model) + classifier + temperatures ----
import subprocess, sys, torch
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=False)
from ultralytics import YOLO

def is_waste_detector(path):
    try:
        m = YOLO(path)
        names = m.names if isinstance(m.names, dict) else dict(enumerate(m.names))
        return len(names) == 1, m
    except Exception:
        return False, None

det = None
cands  = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True), key=len)
cands += sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True), key=len)
for c in cands:
    if os.path.basename(c).startswith('weights_'): continue
    ok, m = is_waste_detector(c)
    if ok:
        det = m; print('detector:', c); break
assert det is not None, 'attach the fine-tuned 1-class detector (sc8 det/weights/best.pt)'

BB = 'convnext_small.fb_in22k_ft_in1k'
wpath = T.find_one(f'weights_{BB}.pt')
clf = T.MultiTaskNet(BB).to(T.DEVICE)
clf.load_state_dict(torch.load(wpath, map_location=T.DEVICE)); clf.eval()
print('classifier:', wpath)

cal_path = T.find_one('calibration.json', required=False)
TEMPS = json.load(open(cal_path)) if cal_path else None
print('temperatures:', 'loaded' if TEMPS else 'none (using T=1)')

DATASET_ROOT = '/kaggle/input/datasets/fbanabil/waste-10class/Dataset_kaggle_release'
ROOT = str(Path(DATASET_ROOT).expanduser().resolve())
_missing_manifests = [name for name in ('train.csv', 'validation.csv', 'test.csv')
                      if not (Path(ROOT) / name).is_file()]
if _missing_manifests:
    raise FileNotFoundError('Missing manifests in DATASET_ROOT=' + ROOT + ': '
                            + ', '.join(_missing_manifests))
print('dataset root:', ROOT)
_, _, df_te = T.load_manifests(ROOT)
df_te = df_te.reset_index(drop=True)

In [ ]:
# ---- full pipeline per item: detect -> crop -> classify all views -> aggregate -> decide ----
def norm_box_from_xyxy(shape, xyxy):
    H, W = shape[:2]
    x1, y1, x2, y2 = [float(v) for v in xyxy]
    return (x1 / W, y1 / H, (x2 - x1) / W, (y2 - y1) / H)

def apply_T(z, head):
    if not TEMPS: return z
    t = TEMPS[head]['T']
    return z / np.array(t, dtype=np.float64) if isinstance(t, list) else z / float(t)

def run_item(item_id):
    rows = df_te[df_te.item_id == item_id]
    zs = {k: [] for k in ['material', 'condition', 'route', 'threat']}
    first_vis = None
    for _, r in rows.iterrows():
        im = T.load_rgb(r['abspath'])
        res = det.predict(r['abspath'], verbose=False, conf=0.25)[0]
        if len(res.boxes) == 0: continue
        b = res.boxes[int(res.boxes.conf.argmax())]
        nbox = norm_box_from_xyxy(im.shape, b.xyxy[0].tolist())
        crop = T.gray_world(T.object_crop(im, nbox, pad=0.2))
        x = T.apply_tf(T.eval_tf, crop).unsqueeze(0).to(T.DEVICE)
        with torch.no_grad():
            o = clf(x)
            o2 = clf(torch.flip(x, dims=[3]))
            o = {k: ((o[k] + o2[k]) / 2).float().cpu().numpy()[0] for k in o}
        for k in zs: zs[k].append(o[k])
        if first_vis is None:
            first_vis = (im, b.xyxy[0].tolist(), crop, float(b.conf[0]))
    if not zs['material']: return None
    z = {k: np.mean(np.stack(v), axis=0) for k, v in zs.items()}
    mprob = T.softmax_np(apply_T(z['material'][None], 'material'))
    cprob = T.sigmoid_np(np.atleast_1d(apply_T(z['condition'], 'condition')))
    tprob = T.corn_class_probs(torch.tensor(apply_T(z['threat'][None], 'threat'))).numpy()
    rprob = T.softmax_np(apply_T(z['route'][None], 'route'))
    dec = T.decide(mprob, cprob, tprob, route_probs=rprob)
    r0 = rows.iloc[0]
    return {'item': item_id, 'vis': first_vis, 'n_views': len(zs['material']),
            'mat_pred': T.MATERIALS[int(mprob[0].argmax())], 'mat_conf': float(mprob[0].max()),
            'cond_pred': 'Poor' if cprob[0] > 0.5 else 'Good',
            'route_dec': str(dec['route'][0]), 'threat_dec': int(dec['threat'][0]),
            'why': dec['expl'][0],
            'gt_mat': T.MATERIALS[int(r0['y_material'])],
            'gt_cond': 'Poor' if r0['y_condition'] > 0.5 else 'Good',
            'gt_route': T.ROUTES[int(r0['y_route'])], 'gt_threat': int(r0['y_threat'])}

results = []
for it in df_te.item_id.drop_duplicates():
    out = run_item(it)
    if out: results.append(out)
print('pipeline ran on', len(results), 'items')

In [ ]:
# ---- select showcase items and render the annotated strip ----

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter

SHOWCASE_ITEM_IDS = ['item_001', 'item_101', 'item_049']
EXCLUDED_SHOWCASE_ITEM_IDS = ['item_218']
item_counts = Counter(s['item'] for s in results)
duplicates = sorted(item for item, count in item_counts.items() if count != 1)
if duplicates:
    raise RuntimeError('Duplicate pipeline item results: ' + ', '.join(duplicates))
missing = [item for item in SHOWCASE_ITEM_IDS if item not in item_counts]
if missing:
    raise RuntimeError('Requested showcase items have no successful pipeline prediction: '
                       + ', '.join(missing) + '. Check the test manifest and detections; '
                       'no replacement examples will be selected.')
by_item = {s['item']: s for s in results}

def showcase_tagline(s):
    
    if s['gt_mat'] in ('battery', 'sharps', 'e_waste') and s['gt_route'] == 'Special':
        return 'hazard -> Special' if s['route_dec'] == 'Special' else 'hazard (GT Special)'
    if s['gt_mat'] == 'glass':
        return 'glass + ' + s['gt_cond']
    return s['gt_mat'].replace('_', ' ')

show = [dict(by_item[item], tagline=showcase_tagline(by_item[item]))
        for item in SHOWCASE_ITEM_IDS]
assert [s['item'] for s in show] == SHOWCASE_ITEM_IDS
assert not set(SHOWCASE_ITEM_IDS).intersection(EXCLUDED_SHOWCASE_ITEM_IDS)
print('showcasing:', [(s['tagline'], s['item']) for s in show])

n = len(show)
fig, axes = plt.subplots(n, 3, figsize=(9.4, 2.75 * n), width_ratios=[1.2, 1, 1.35])
for i, s in enumerate(show):
    im, xyxy, crop, dconf = s['vis']
    row = axes[i] if n > 1 else axes
    ax = row[0]; ax.imshow(im)
    x1, y1, x2, y2 = xyxy
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color='#E65100', lw=2.2))
    ax.set_title(f"{s['item']}  det {dconf:.2f}", fontsize=9); ax.axis('off')
    ax = row[1]; ax.imshow(crop); ax.set_title(f"crop ({s['n_views']} views)", fontsize=9); ax.axis('off')
    ax = row[2]; ax.axis('off')
    ok = s['route_dec'] == s['gt_route']
    txt = (f"{s['tagline']}\n"
           f"material: {s['mat_pred']} ({s['mat_conf']:.2f})  [GT {s['gt_mat']}]\n"
           f"condition: {s['cond_pred']}  [GT {s['gt_cond']}]\n"
           f"threat: level {s['threat_dec']}  [GT {s['gt_threat']}]\n"
           f"route: {s['route_dec']}  [GT {s['gt_route']}]  {'OK' if ok else 'X'}")
    ax.text(0.02, 0.95, txt, va='top', fontsize=9.5, family='monospace',
            bbox=dict(boxstyle='round,pad=0.45', fc='#F5F7FA', ec='#B0BEC5'))
fig.tight_layout()
fig.savefig(f'{OUT}/qualitative_demo.png', dpi=600, bbox_inches='tight')
plt.close(fig)
print('wrote qualitative_demo.png')

import pandas as pd
from pathlib import Path
from PIL import Image
import hashlib


selected_taglines = {s['item']: s['tagline'] for s in show}
prediction_rows = [
    dict({k: v for k, v in s.items() if k not in ('vis', 'why', 'tagline')},
         tagline=selected_taglines.get(s['item'], ''))
    for s in results
]
pd.DataFrame(prediction_rows).to_csv(f'{OUT}/pipeline_predictions.csv', index=False)
print('wrote pipeline_predictions.csv (all', len(results), 'test items)')


figure_path = Path(OUT) / 'qualitative_demo.png'
with Image.open(figure_path) as exported_image:
    raster_size = list(exported_image.size)
qualitative_manifest = {
    'schema_version': 1,
    'producer': 'SC22',
    'selected_item_ids': SHOWCASE_ITEM_IDS,
    'excluded_item_ids': EXCLUDED_SHOWCASE_ITEM_IDS,
    'figure_file': figure_path.name,
    'figure_sha256': hashlib.sha256(figure_path.read_bytes()).hexdigest(),
    'raster_size': raster_size,
    'rows': [{'item_id': s['item'], 'tagline': s['tagline']} for s in show],
}
(Path(OUT) / 'qualitative_manifest.json').write_text(
    json.dumps(qualitative_manifest, indent=2) + '\n', encoding='utf-8')
print('wrote qualitative_manifest.json:', qualitative_manifest['selected_item_ids'])


## Outputs

- `qualitative_demo.png`: the three selected showcase items.
- `pipeline_predictions.csv`: all successful per-item end-to-end predictions.
- `qualitative_manifest.json`: selected item identifiers, image dimensions, and figure fingerprint.

Download the complete output into `Results/SC22/`. Preserve the manifest with its matching image.
